In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = '/content/drive/MyDrive/RecSys'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f"Working in: {os.getcwd()}")

Mounted at /content/drive
Working in: /content/drive/MyDrive/RecSys


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import torch, numpy, pandas
print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None — go to Runtime > Change runtime type > T4 GPU'}")

PyTorch: 2.10.0+cu128
GPU: Tesla T4


In [4]:
from google.colab import files

# Upload all 3 files when the dialog opens:
#   build_realistic_temporal_intent_dataset.py
#   train_temporal_multi_intent_transformer.py
#   run_ablations.py
uploaded = files.upload()

import shutil
for fname in uploaded:
    shutil.move(fname, f'{WORK_DIR}/{fname}')
    print(f"Saved: {fname}")

Saving build_realistic_temporal_intent_dataset_v2 (1).py to build_realistic_temporal_intent_dataset_v2 (1).py
Saved: build_realistic_temporal_intent_dataset_v2 (1).py


In [4]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, 'build_realistic_temporal_intent_dataset.py'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("ERRORS:", result.stderr)

Building videos...
  Videos: 15000
Building users with intent profiles...
  Users: 5000
Building watch history (this takes ~30s)...
  user 500/5000...
  user 1000/5000...
  user 1500/5000...
  user 2000/5000...
  user 2500/5000...
  user 3000/5000...
  user 3500/5000...
  user 4000/5000...
  user 4500/5000...
  user 5000/5000...
Saving watch history...
  Interactions: 578417
Building sequence windows (NPZ)...
  Windows: 573417

✅ Done! Summary:
{
  "n_users": 5000,
  "n_videos": 15000,
  "n_interactions": 578417,
  "n_windows": 573417,
  "n_intents_max": 4,
  "n_intent_clusters": 8,
  "topic_count": 18,
  "max_seq_len": 50,
  "split_train": 455737,
  "split_val": 57595,
  "split_test": 60085,
  "experimental_goal": "Temporal multi-intent: baseline < time-aware < multi-intent < time+multi-intent"
}



In [3]:
!ls

'build_realistic_temporal_intent_dataset (1).py'
 build_realistic_temporal_intent_dataset.py
 realistic_dataset
'run_ablations (1).py'
 run_ablations.py
'train_temporal_multi_intent_transformer (1).py'
 train_temporal_multi_intent_transformer.py


In [4]:
import json, numpy as np

meta = json.load(open('realistic_dataset/metadata.json'))
print(f"Users: {meta['n_users']:,}")
print(f"Videos: {meta['n_videos']:,}")
print(f"Interactions: {meta['n_interactions']:,}")
print(f"Windows — train: {meta['split_train']:,} | val: {meta['split_val']:,} | test: {meta['split_test']:,}")

data = np.load('realistic_dataset/temporal_intent_windows.npz')
print(f"\nNPZ arrays: {list(data.keys())}")

Users: 5,000
Videos: 15,000
Interactions: 578,417
Windows — train: 455,737 | val: 57,595 | test: 60,085

NPZ arrays: ['user_id', 'hist_item_id', 'hist_hour_of_day', 'hist_time_bucket', 'hist_topic_id', 'hist_intent_slot', 'hist_intent_act', 'hist_watch_ratio', 'target_item_id', 'target_intent', 'split_code']


In [5]:
!python train_temporal_multi_intent_transformer.py \
    --data-npz  realistic_dataset/temporal_intent_windows.npz \
    --out-model models/baseline.pt \
    --model-type baseline \
    --n-intents 4 \
    --d-model   128 \
    --d-time    32 \
    --epochs    10 \
    --batch-size 256

Device: cuda | Model type: baseline
Dataset: 15000 items | train=455737 val=57595 test=60085
Parameters: 2,323,328
Epoch 01 | loss=0.6063 | ACC@1=0.0025 HR@10=0.0141 NDCG@10=0.0067 MRR@10=0.0046
Epoch 02 | loss=0.1041 | ACC@1=0.0010 HR@10=0.0186 NDCG@10=0.0082 MRR@10=0.0051
Epoch 03 | loss=0.0770 | ACC@1=0.0022 HR@10=0.0183 NDCG@10=0.0084 MRR@10=0.0055
Epoch 04 | loss=0.0649 | ACC@1=0.0023 HR@10=0.0212 NDCG@10=0.0099 MRR@10=0.0065
Epoch 05 | loss=0.0586 | ACC@1=0.0027 HR@10=0.0215 NDCG@10=0.0100 MRR@10=0.0067
Epoch 06 | loss=0.0535 | ACC@1=0.0019 HR@10=0.0201 NDCG@10=0.0092 MRR@10=0.0060
Epoch 07 | loss=0.0508 | ACC@1=0.0023 HR@10=0.0215 NDCG@10=0.0100 MRR@10=0.0065
Epoch 08 | loss=0.0480 | ACC@1=0.0024 HR@10=0.0200 NDCG@10=0.0093 MRR@10=0.0062
Epoch 09 | loss=0.0463 | ACC@1=0.0021 HR@10=0.0206 NDCG@10=0.0095 MRR@10=0.0062
Epoch 10 | loss=0.0443 | ACC@1=0.0021 HR@10=0.0214 NDCG@10=0.0098 MRR@10=0.0063

TEST | ACC@1=0.0020 HR@10=0.0196 NDCG@10=0.0090 MRR@10=0.0059
Results saved to model

In [6]:
!python train_temporal_multi_intent_transformer.py \
    --data-npz  realistic_dataset/temporal_intent_windows.npz \
    --out-model models/baseline.pt \
    --model-type time_only \
    --n-intents 4 \
    --d-model   128 \
    --d-time    32 \
    --epochs    10 \
    --batch-size 256

Device: cuda | Model type: time_only
Dataset: 15000 items | train=455737 val=57595 test=60085
Parameters: 2,345,472
Epoch 01 | loss=0.5849 | ACC@1=0.0000 HR@10=0.0169 NDCG@10=0.0065 MRR@10=0.0035
Epoch 02 | loss=0.1013 | ACC@1=0.0022 HR@10=0.0196 NDCG@10=0.0091 MRR@10=0.0060
Epoch 03 | loss=0.0759 | ACC@1=0.0018 HR@10=0.0172 NDCG@10=0.0080 MRR@10=0.0053
Epoch 04 | loss=0.0643 | ACC@1=0.0024 HR@10=0.0202 NDCG@10=0.0094 MRR@10=0.0062
Epoch 05 | loss=0.0584 | ACC@1=0.0027 HR@10=0.0205 NDCG@10=0.0098 MRR@10=0.0066
Epoch 06 | loss=0.0535 | ACC@1=0.0019 HR@10=0.0205 NDCG@10=0.0094 MRR@10=0.0061
Epoch 07 | loss=0.0509 | ACC@1=0.0024 HR@10=0.0219 NDCG@10=0.0101 MRR@10=0.0066
Epoch 08 | loss=0.0483 | ACC@1=0.0024 HR@10=0.0220 NDCG@10=0.0102 MRR@10=0.0067
Epoch 09 | loss=0.0468 | ACC@1=0.0024 HR@10=0.0223 NDCG@10=0.0103 MRR@10=0.0067
Epoch 10 | loss=0.0450 | ACC@1=0.0024 HR@10=0.0233 NDCG@10=0.0106 MRR@10=0.0068

TEST | ACC@1=0.0021 HR@10=0.0221 NDCG@10=0.0100 MRR@10=0.0065
Results saved to mode

In [7]:
!python train_temporal_multi_intent_transformer.py \
    --data-npz  realistic_dataset/temporal_intent_windows.npz \
    --out-model models/baseline.pt \
    --model-type intent_only \
    --n-intents 4 \
    --d-model   128 \
    --d-time    32 \
    --epochs    10 \
    --batch-size 256

Device: cuda | Model type: intent_only
Dataset: 15000 items | train=455737 val=57595 test=60085
Parameters: 2,377,253
Epoch 01 | loss=0.2251 | ACC@1=0.0019 HR@10=0.0198 NDCG@10=0.0091 MRR@10=0.0059 IntentACC=0.0952
Epoch 02 | loss=0.0815 | ACC@1=0.0025 HR@10=0.0194 NDCG@10=0.0094 MRR@10=0.0064 IntentACC=0.1066
Epoch 03 | loss=0.0655 | ACC@1=0.0023 HR@10=0.0187 NDCG@10=0.0089 MRR@10=0.0060 IntentACC=0.0562
Epoch 04 | loss=0.0582 | ACC@1=0.0028 HR@10=0.0191 NDCG@10=0.0092 MRR@10=0.0063 IntentACC=0.0562
Epoch 05 | loss=0.0538 | ACC@1=0.0026 HR@10=0.0220 NDCG@10=0.0102 MRR@10=0.0067 IntentACC=0.0575
Epoch 06 | loss=0.0511 | ACC@1=0.0027 HR@10=0.0214 NDCG@10=0.0099 MRR@10=0.0065 IntentACC=0.3783
Epoch 07 | loss=0.0489 | ACC@1=0.0027 HR@10=0.0238 NDCG@10=0.0108 MRR@10=0.0070 IntentACC=0.0562
Epoch 08 | loss=0.0480 | ACC@1=0.0020 HR@10=0.0232 NDCG@10=0.0101 MRR@10=0.0063 IntentACC=0.0565
Epoch 09 | loss=0.0471 | ACC@1=0.0023 HR@10=0.0240 NDCG@10=0.0107 MRR@10=0.0068 IntentACC=0.0563
Epoch 10 

In [6]:
!python train_temporal_multi_intent_transformer.py \
    --data-npz  realistic_dataset/temporal_intent_windows.npz \
    --out-model models/time_intent.pt \
    --model-type time_intent \
    --n-intents 4 \
    --d-model   256 \
    --d-time    64 \
    --epochs    10 \
    --batch-size 256

Device: cuda | Model type: time_intent
Dataset: 15000 items | train=455737 val=57595 test=60085
Parameters: 5,739,205
Epoch 01 | loss=0.2056 | ACC@1=0.0023 HR@10=0.0148 NDCG@10=0.0073 MRR@10=0.0051 IntentACC=0.2780
Epoch 02 | loss=0.0782 | ACC@1=0.0015 HR@10=0.0187 NDCG@10=0.0083 MRR@10=0.0052 IntentACC=0.6778
Epoch 03 | loss=0.0641 | ACC@1=0.0017 HR@10=0.0177 NDCG@10=0.0083 MRR@10=0.0054 IntentACC=0.6073
Epoch 04 | loss=0.0571 | ACC@1=0.0028 HR@10=0.0233 NDCG@10=0.0109 MRR@10=0.0072 IntentACC=0.6780
Epoch 05 | loss=0.0538 | ACC@1=0.0020 HR@10=0.0189 NDCG@10=0.0088 MRR@10=0.0058 IntentACC=0.2842
Epoch 06 | loss=0.0509 | ACC@1=0.0028 HR@10=0.0216 NDCG@10=0.0105 MRR@10=0.0072 IntentACC=0.1026
Epoch 07 | loss=0.0481 | ACC@1=0.0026 HR@10=0.0230 NDCG@10=0.0107 MRR@10=0.0071 IntentACC=0.6773
Epoch 08 | loss=0.0461 | ACC@1=0.0022 HR@10=0.0246 NDCG@10=0.0110 MRR@10=0.0070 IntentACC=0.6755
Epoch 09 | loss=0.0449 | ACC@1=0.0025 HR@10=0.0250 NDCG@10=0.0113 MRR@10=0.0073 IntentACC=0.6685
Epoch 10 

In [6]:
!python build_realistic_temporal_intent_dataset_v2.py

Output dir: realistic_dataset_v2
Scale: 5000 users x 3000 videos, 150-500 events/user
Building videos...
  Videos: 3000
Building users with intent profiles...
  Users: 5000
Building watch history (this takes ~30s)...
  user 500/5000...
  user 1000/5000...
  user 1500/5000...
  user 2000/5000...
  user 2500/5000...
  user 3000/5000...
  user 3500/5000...
  user 4000/5000...
  user 4500/5000...
  user 5000/5000...
Saving watch history...
  Interactions: 1635310
Building sequence windows (NPZ)...
  Windows: 1630310

✅ Done! Summary:
{
  "n_users": 5000,
  "n_videos": 3000,
  "n_interactions": 1635310,
  "n_windows": 1630310,
  "n_intents_max": 4,
  "n_intent_clusters": 8,
  "topic_count": 18,
  "max_seq_len": 50,
  "split_train": 1301252,
  "split_val": 163269,
  "split_test": 165789,
  "version": "v2",
  "experimental_goal": "Temporal multi-intent: baseline < time-aware < multi-intent < time+multi-intent"
}


In [7]:
import json, numpy as np

meta = json.load(open('realistic_dataset_v2/metadata.json'))
print(f"Users:    {meta['n_users']:,}")
print(f"Videos:   {meta['n_videos']:,}")
print(f"Events:   {meta['n_interactions']:,}")
print(f"Train:    {meta['split_train']:,}")
print(f"Val:      {meta['split_val']:,}")
print(f"Test:     {meta['split_test']:,}")

data = np.load('realistic_dataset_v2/temporal_intent_windows.npz')
print(f"\nArrays:   {list(data.keys())}")

Users:    5,000
Videos:   3,000
Events:   1,635,310
Train:    1,301,252
Val:      163,269
Test:     165,789

Arrays:   ['user_id', 'hist_item_id', 'hist_hour_of_day', 'hist_time_bucket', 'hist_topic_id', 'hist_intent_slot', 'hist_intent_act', 'hist_watch_ratio', 'target_item_id', 'target_intent', 'split_code']


In [ ]:
!python train_temporal_multi_intent_transformer.py \
    --data-npz   realistic_dataset_v2/temporal_intent_windows.npz \
    --out-model  models/baseline_v2.pt \
    --model-type baseline \
    --n-intents  4 \
    --d-model    256 \
    --n-layers   3 \
    --d-time     32 \
    --epochs     20 \
    --batch-size 512

Device: cuda | Model type: baseline
Dataset: 3000 items | train=1301252 val=163269 test=165789
Parameters: 3,150,848
Epoch 01 | loss=0.2542 | ACC@1=0.0047 HR@10=0.0559 NDCG@10=0.0257 MRR@10=0.0167
Epoch 02 | loss=0.0836 | ACC@1=0.0172 HR@10=0.1105 NDCG@10=0.0560 MRR@10=0.0397
Epoch 03 | loss=0.0702 | ACC@1=0.0139 HR@10=0.1082 NDCG@10=0.0505 MRR@10=0.0335
Epoch 04 | loss=0.0585 | ACC@1=0.0138 HR@10=0.1118 NDCG@10=0.0539 MRR@10=0.0366
Epoch 05 | loss=0.0526 | ACC@1=0.0158 HR@10=0.1086 NDCG@10=0.0520 MRR@10=0.0352
Epoch 06 | loss=0.0490 | ACC@1=0.0160 HR@10=0.1204 NDCG@10=0.0570 MRR@10=0.0382
Epoch 07 | loss=0.0481 | ACC@1=0.0171 HR@10=0.1241 NDCG@10=0.0591 MRR@10=0.0399
